# Parte 3 — Extensión C: reranking

Se traen 20 fragmentos con bge-m3 (H200, VPN), se reordenan con el reranker `rerank_local_ingles` (`cross-encoder/ms-marco-MiniLM-L6-v2`, verificado 2026-09-19, corre en local) y se toman los k primeros (k = 3 y 5).

**Pregunta:** elige la extensión por el par (Hit Rate, MRR) y las tasas de abstención que mediste, y declara antes de implementarla qué modo de falla esperas que corrija.

El baseline tiene Hit Rate 0,750 y MRR 0,583. En las 8 preguntas respondibles el primer fragmento ya es del paper correcto, pero el fragmento con la respuesta sale más abajo: en el lugar 13 en la pregunta 1, en el 8 en la 6 y en el 3 en la 9 y la 10. Por eso se eligió reranking: se espera que reordene y suba esos fragmentos, y que así mejore el MRR. Se usa el reranker en inglés porque los papers están en inglés.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append("..")

from sentence_transformers import CrossEncoder

from evaluation import posicion_del_primer_acierto, escribir_csv
from rag_pipeline import RagPipeline, EMBEDDING_MODEL

pipeline = RagPipeline()
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
golden = json.loads(Path("../golden_set.json").read_text(encoding="utf-8"))

/home/harold/Documents/maestria/primer semestre/IA generativa y agentes/semana 2/taller 2/taller02-rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10750.43it/s]

In [2]:
def recuperar_con_rerank(pregunta, k):
    candidatos = pipeline.retrieve(pregunta, top_k=20)
    puntajes = reranker.predict([(pregunta, c["text"]) for c in candidatos])
    for c, s in zip(candidatos, puntajes):
        c["rerank"] = float(s)
    candidatos.sort(key=lambda c: c["rerank"], reverse=True)
    return candidatos[:k]

In [3]:
filas = []
for k in [3, 5]:
    for g in golden:
        if not g["documentos_fuente"]:
            continue
        hits = recuperar_con_rerank(g["pregunta"], k)
        pos = posicion_del_primer_acierto(hits, g)
        filas.append({"id": g["id"], "tipo": g["tipo"], "respondible": True, "k": k,
                      "posicion": pos, "hit": pos is not None, "reciprocal_rank": 1 / pos if pos else 0.0,
                      "score_top1": hits[0]["rerank"], "abstuvo": None, "generador": None,
                      "retrieved_ids": [h["chunk_id"] for h in hits]})

escribir_csv(filas, Path("../resultados_rerank.csv"), EMBEDDING_MODEL + " + ms-marco-MiniLM-L6-v2")
print(len(filas), "filas en resultados_rerank.csv")

16 filas en resultados_rerank.csv


Comparación con el baseline (los dos desde sus CSV):

In [4]:
import csv


def leer(nombre):
    with open(f"../{nombre}", encoding="utf-8") as f:
        return [r for r in csv.DictReader(f) if r["respondible"] == "True"]


base = leer("resultados.csv")
rerank = leer("resultados_rerank.csv")

print(f"{'':<10} {'k':>2} {'Hit Rate':>9} {'MRR':>6}")
for nombre, filas_csv in [("baseline", base), ("rerank", rerank)]:
    for k in ["3", "5"]:
        del_k = [r for r in filas_csv if r["k"] == k]
        hit = sum(r["hit"] == "True" for r in del_k) / len(del_k)
        mrr = sum(float(r["reciprocal_rank"]) for r in del_k) / len(del_k)
        print(f"{nombre:<10} {k:>2} {hit:>9.3f} {mrr:>6.3f}")

            k  Hit Rate    MRR
baseline    3     0.750  0.583
baseline    5     0.750  0.583
rerank      3     0.625  0.562
rerank      5     0.750  0.594


In [5]:
print("posición del primer acierto con k=5")
print(f"{'id':>3} {'tipo':<12} {'baseline':>9} {'rerank':>7}")
for b, r in zip([x for x in base if x["k"] == "5"], [x for x in rerank if x["k"] == "5"]):
    print(f"{b['id']:>3} {b['tipo']:<12} {b['posicion'] or '-':>9} {r['posicion'] or '-':>7}")

posición del primer acierto con k=5
 id tipo          baseline  rerank
  1 simple               -       -
  2 simple               1       1
  3 simple               1       1
  4 multi-chunk          1       1
  5 multi-chunk          1       1
  6 multi-chunk          -       2
  9 adversarial          3       4
 10 simple               3       -


Qué hizo el reranker en las preguntas que cambiaron (lugar después del reranking y lugar que tenía en la búsqueda densa):

In [3]:
from evaluation import acierta

golden_por_id = {g["id"]: g for g in golden}
for id_ in [1, 6, 9, 10]:
    g = golden_por_id[id_]
    orden = recuperar_con_rerank(g["pregunta"], 20)
    densos = [c["chunk_id"] for c in pipeline.retrieve(g["pregunta"], top_k=20)]
    print("pregunta", id_)
    for pos, c in enumerate(orden, start=1):
        marca = "<- fragmento anotado" if acierta(c, g) else ""
        if pos <= 3 or marca:
            print(f"  rerank {pos:>2} (denso {densos.index(c['chunk_id']) + 1:>2})  {c['rerank']:6.2f}  {c['chunk_id']:<34} {marca}")
    print()

pregunta 1
  rerank  1 (denso 18)    2.74  wei-2022-chain-of-thought-0044     
  rerank  2 (denso  3)    2.31  wei-2022-chain-of-thought-0020     
  rerank  3 (denso  2)    0.94  wei-2022-chain-of-thought-0055     
  rerank 16 (denso 13)   -5.23  wei-2022-chain-of-thought-0009     <- fragmento anotado



pregunta 6
  rerank  1 (denso  1)    4.89  es-2023-ragas-0012                 
  rerank  2 (denso  8)    3.41  es-2023-ragas-0005                 <- fragmento anotado
  rerank  3 (denso  4)    2.70  es-2023-ragas-0002                 



pregunta 9
  rerank  1 (denso  7)    2.74  ouyang-2022-instructgpt-0004       
  rerank  2 (denso  8)    2.67  brown-2020-gpt3-0045               
  rerank  3 (denso 16)    2.24  brown-2020-gpt3-0010               
  rerank  4 (denso  3)    2.20  brown-2020-gpt3-0017               <- fragmento anotado



pregunta 10
  rerank  1 (denso  1)    2.13  ouyang-2022-instructgpt-0024       
  rerank  2 (denso  5)    2.08  ouyang-2022-instructgpt-0006       
  rerank  3 (denso  8)    1.63  ouyang-2022-instructgpt-0035       
  rerank  6 (denso  3)    0.44  ouyang-2022-instructgpt-0004       <- fragmento anotado



**Resultado:** el reranking no mejoró en general. Con k=5 el Hit Rate quedó igual (0,750) y el MRR subió un poco (de 0,583 a 0,594). Con k=3 bajaron los dos (Hit Rate de 0,750 a 0,625 y MRR de 0,583 a 0,562).

- Pregunta 6: mejoró. El fragmento correcto subió del lugar 8 al 2.
- Pregunta 1: siguió sin acierto. El fragmento con el dato bajó del lugar 13 al 16.
- Pregunta 9: empeoró. Subió al primer lugar un fragmento de InstructGPT que habla de GPT-3 175B, y el correcto bajó del 3 al 4.
- Pregunta 10: empeoró. Subió otro fragmento de InstructGPT que también compara con GPT-3, y el correcto bajó del 3 al 6.

Probablemente no mejoró porque hay varios fragmentos que hablan del mismo tema, y el reranker a veces sube uno que no tiene la frase anotada, así que cuenta como fallo aunque hable de lo mismo. Además, las cifras que publica este reranker son de búsquedas en internet (MS MARCO), no de papers, así que no había garantía de que funcionara igual con este corpus.